In [10]:
# Importa tudo

from selenium import webdriver
from selenium.webdriver.support.select import Select
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException
from selenium.common.exceptions import TimeoutException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager

#Bibliotecas de Sistema
import time
import re
import csv
import os
import psutil
from bs4 import BeautifulSoup
from eproc_driver import eproc as eproc
import sqlite3
from pathlib import Path
import io
import pandas as pd
from contextlib import closing

#bibliotecas de configuração
import pyotp
import configparser
import keyring

#bibliotecas de automação
import pyperclip
import pyautogui

#Bibliotecas de IA
from gemini import gemini as gemini
import ollama

# Funções

def inserir_pedido(pedido):
    conn = sqlite3.connect('movimentos.db')
    with closing(conn.cursor()) as cursor:
        cursor.execute(
            "INSERT INTO pedidos (resumo) VALUES (?)",
            (pedido,)
        )
        conn.commit()
    conn.close()

In [11]:
def lista_tipos_pedidos():
    conn = sqlite3.connect('movimentos.db')
    with closing(conn.cursor()) as cursor:
        cursor.execute("SELECT id, resumo FROM pedidos")
        pedidos_lista = [f"ID: {row[0]}. RESUMO : {row[1]}" for row in cursor.fetchall() if row[1]]
    conn.close()
    tipos_pedidos = "\n".join(pedidos_lista)
    return tipos_pedidos

In [16]:
# Lista os tipos de pedidos já cadastrados no banco de dados
tipos_pedidos = lista_tipos_pedidos()
print(tipos_pedidos)

ID: 1. RESUMO : Pedido de desistência do processo
ID: 2. RESUMO : Informação de que a parte está ciente.
ID: 3. RESUMO : Solicita a inclusão da parte executada em cadastros de inadimplentes para fins de cobrança de dívida tributária.
ID: 4. RESUMO : Pedido de desconsideração do valor irrisório bloqueado e regular prosseguimento do processo.
ID: 5. RESUMO : O município solicita um prazo estendido para realizar o protesto do título, visando à extinção da execução fiscal de baixo valor.
ID: 6. RESUMO : Solicita-se a desconsideração do valor irrisório bloqueado e regular prosseguimento do processo.
ID: 7. RESUMO : Solicita-se a penhora online via Sisbajud, modalidade reiterada, para identificar e bloquear valores em contas da executada.


In [15]:
pedido = "Solicita-se a penhora online via Sisbajud, modalidade reiterada, para identificar e bloquear valores em contas da executada."
inserir_pedido(pedido)

In [6]:
conn = sqlite3.connect('movimentos.db')
with closing(conn.cursor()) as cursor:
    cursor.execute("SELECT id, documentos FROM movimentos WHERE documentos LIKE '%*PET*%' AND resumo IS NULL")
    documentos_pet = cursor.fetchall()
conn.close()
print(documentos_pet)

[]


In [12]:
for peticao in documentos_pet:
    pet_id, documentos = peticao
    resumo = ollama_resumo(documentos)
    print(f"ID: {pet_id}, Resumo: {resumo}")
    conn = sqlite3.connect('movimentos.db')
    with closing(conn.cursor()) as cursor:
        cursor.execute(
            "UPDATE movimentos SET resumo = ? WHERE id = ?",
            (resumo, pet_id)
        )
        conn.commit()
    conn.close()